In [37]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver, InMemorySaver  
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from typing import Dict, Any, Optional
import re
from dotenv import load_dotenv
import os
import requests
load_dotenv()
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")



In [38]:
class State(BaseModel):
    #name: str = Field(..., description="The name of the state")
    #response_txt: str = Field(..., description="The response text of the state")
    city: str = Field(..., description="The city for which to get the weather")

llm = init_chat_model(
    model = 'gpt-5-nano',
    tools = []
)
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"


def connect_weatherapi(city: str, units: str = "metric", timeout: int = 10) -> Dict[str, Any]:
    """Return the weather information for a given city."""
    params = {"q": city, "units": units, "appid": WEATHER_API_KEY}
    resp = requests.get(BASE_URL, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def extract_city_from_message(message_text: str) -> str:
    match = re.search(r'weather in ([A-Za-z\s]+)', message_text, re.IGNORECASE)
    if match:
        return match.group(1).strip().rstrip('?.!,')
    return "London"

def weather_node(state: dict) -> dict:
    message = state["messages"][-1]
    city = extract_city_from_message(message.content)
    return {"weather": connect_weatherapi(city)}

print(connect_weatherapi("London"))

/Users/vinothkumarkothandapani/Downloads/AIProject/MyProject/.venv/lib/python3.13/site-packages/langchain/chat_models/base.py:477: UserWarning: WARNING! tools is not default parameter.
                tools was transferred to model_kwargs.
                Please confirm that tools is what you intended.
  return _init_chat_model_helper(


{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 11.04, 'feels_like': 10.11, 'temp_min': 9.98, 'temp_max': 11.86, 'pressure': 1019, 'humidity': 73, 'sea_level': 1019, 'grnd_level': 1015}, 'visibility': 10000, 'wind': {'speed': 3.6, 'deg': 270}, 'clouds': {'all': 100}, 'dt': 1775086404, 'sys': {'type': 2, 'id': 2075535, 'country': 'GB', 'sunrise': 1775108029, 'sunset': 1775154845}, 'timezone': 3600, 'id': 2643743, 'name': 'London', 'cod': 200}


In [43]:
from langchain_community import memory


graph = StateGraph(dict)
graph.add_node("weather_nodename", weather_node)
graph.add_edge(START, "weather_nodename")
graph.add_edge("weather_nodename", END)
config = ({"configurable":{"thread_id":"mytthreadid"}})
memory = MemorySaver()
response = graph.compile(checkpointer=memory)
response.invoke({"messages": [HumanMessage(content="What is the weather in Chennai?")]},config=config)

{'weather': {'coord': {'lon': 80.2785, 'lat': 13.0878},
  'weather': [{'id': 701,
    'main': 'Mist',
    'description': 'mist',
    'icon': '50n'}],
  'base': 'stations',
  'main': {'temp': 26.46,
   'feels_like': 26.46,
   'temp_min': 25.05,
   'temp_max': 26.65,
   'pressure': 1010,
   'humidity': 78,
   'sea_level': 1010,
   'grnd_level': 1010},
  'visibility': 5000,
  'wind': {'speed': 2.06, 'deg': 270},
  'clouds': {'all': 40},
  'dt': 1775086492,
  'sys': {'type': 2,
   'id': 2104103,
   'country': 'IN',
   'sunrise': 1775090070,
   'sunset': 1775134219},
  'timezone': 19800,
  'id': 1264527,
  'name': 'Chennai',
  'cod': 200}}